## 02 - Chunk prepared documents
Load prepared documents/queries, apply a chunker, and write `passages.jsonl` for retrieval experiments.

In [ ]:
# Robustly set repo root and import path
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "src").exists():
        ROOT = cand
        break
else:
    raise RuntimeError("Could not locate repo root containing src/.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Repo root: {ROOT}")


### Configure inputs and chunker
Point to `documents.jsonl`, choose a chunker, set params, and pick the output path for `passages.jsonl`.

In [ ]:
# ---- User choices ----
documents_path = ROOT / "data" / "processed" / "poquad" / "example" / "documents.jsonl"
chunker_name = "fixed_size"  # fixed_size | passage | text_tiling
chunker_params = {"chunk_size": 300, "overlap": 40}  # override defaults as needed
passages_output_path = ROOT / "data" / "processed" / "poquad" / "example" / "passages.jsonl"
overwrite = False

### Load documents
Read the prepared documents JSONL and preview the first record.

In [ ]:
from src.data_loader.core.schemas import load_document_records_jsonl

documents = load_document_records_jsonl(str(documents_path))
print(f"Loaded {len(documents)} documents from {documents_path}")
print("First document:\n", documents[0].to_dict() if documents else {})

### Chunk documents
Apply the chosen chunker and collect `PassageRecord` entries with `parentId` pointing to the source document.

In [ ]:
from src.chunking import get_chunker
from src.data_loader.core.schemas import PassageRecord

chunker = get_chunker(chunker_name, chunker_params)
doc_texts = [doc.contents for doc in documents]
doc_metas = [{"doc_id": doc.doc_id, **(doc.metadata or {})} for doc in documents]

chunks = chunker.split_text(doc_texts, documents_meta=doc_metas)
passages = []
for ch in chunks:
    meta = ch.metadata or {}
    parent_id = str(meta.get("doc_id", ""))
    passages.append(
        PassageRecord(
            passage_id=str(ch.chunk_id),
            contents=ch.text,
            parent_id=parent_id,
            metadata=meta,
        )
    )

total_chunks = len(chunks)

print(f"Produced {total_chunks} passages across {len(documents)} documents")
print("First passage:\n", passages[0].to_dict() if passages else {})

### Save passages
Write `passages.jsonl` with passage IDs, text, metadata, and parent document IDs.

In [ ]:
import os
import json
from datetime import datetime, timezone
from src.data_loader.core.schemas import save_passage_records_jsonl

os.makedirs(passages_output_path.parent, exist_ok=True)
base, ext = os.path.splitext(str(passages_output_path))

if overwrite:
    final_path = str(passages_output_path)
else:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    final_path = f"{base}_{ts}{ext}"

meta_path = final_path.rsplit(".", 1)[0] + ".meta.json"

save_passage_records_jsonl(passages, final_path)
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "documents_path": str(documents_path),
            "output_path": final_path,
            "chunker_name": chunker_name,
            "chunker_params": chunker_params,
            "overwrite": overwrite,
            "document_count": len(documents),
            "chunk_count": len(passages),
            "generated_at": datetime.now(timezone.utc).isoformat(),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved passages to {final_path}")
print(f"Saved metadata to {meta_path}")